In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Building Customer Churn Analysis Master tables with aggregations and business use case for dashboard

In [0]:
@dlt.table(
    table_properties={
        "quality": "gold",
    },
    name = "final_churn_analysis",
    comment = "This gold table contains the churn analysis for customers"
)
def customer_churn_analysis():
    dim_customers = spark.read.table("deltalake.new_silver.customers")
    dim_products = spark.read.table("deltalake.new_silver.products")
    fact_transaction = spark.read.table("deltalake.new_silver.fact_transaction")

    # filter only active products 
    active_products = dim_products.filter(expr("__END_AT IS NULL"))

    # calculate customer tenure (in months) & age
    customer_tenure = dim_customers.withColumns({
        "customer_tenure_months" : (months_between(current_date(), col("registration_date"))).cast(IntegerType()),
        "customer_age_years": (datediff(current_date(), col("date_of_birth")) / 365).cast(IntegerType())
    })

    # Aggregate transaction data per customer for active products

    # active_product_transaction = fact_transaction.join(active_products,["product_id"], "inner")

    customer_transaction_summary = fact_transaction.groupBy("customer_id").agg(
        countDistinct("policy_id").alias("total_policies"),
        sum("premium_amount").alias("total_premium_paid"),
        countDistinct(when(upper(col("policy_status")) == "ACTIVE", col("policy_id"))).alias("active_policies"),
        countDistinct(when(upper(col("policy_status")) == "LAPSED", col("policy_id"))).alias("lapsed_policies"),
        max(col("last_payment_date")).alias("last_payment_date"),
        max(col("renewal_date")).alias("last_renewal_date")).withColumns({
        "days_since_last_payment": date_diff(current_date(), col("last_payment_date")),
        "days_since_last_renewal": date_diff(current_date(), col("last_renewal_date"))
    })

    #Form Gold Table

    gold_df = customer_tenure.alias("c") \
        .join(customer_transaction_summary.alias("t"),
              col("c.customer_id") == col("t.customer_id"), "left") \
        .select(
            col("c.customer_id"),
            col("c.full_name"),
            col("c.email_domain"),
            col("c.gender"),
            col("c.registration_date"),
            col("c.customer_tenure_months"),
            col("c.customer_age_years"),
            col("t.total_policies"),
            col("t.total_premium_paid"),
            col("t.active_policies"),
            col("t.lapsed_policies"),
            col("t.days_since_last_payment"),
            col("t.days_since_last_renewal")
    )
    
    return gold_df